In [12]:
import os
from dotenv import load_dotenv

# Load .env from the project root (two directories up)
load_dotenv(dotenv_path="../../.env")

print("✅ Environment loaded successfully!")
print(f"HF_TOKEN loaded: {'HF_TOKEN' in os.environ}")
print(f"Current directory: {os.getcwd()}")

✅ Environment loaded successfully!
HF_TOKEN loaded: True


FileNotFoundError: [Errno 2] No such file or directory

In [10]:
os.environ['HF_TOKEN']=os.getenv("HF_TOKEN")

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings


In [10]:
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

No sentence-transformers model found with name sentence-transformers/all-MiniLM-L6-v2. Creating a new one with mean pooling.


OSError: Could not find a suitable TLS CA certificate bundle, invalid path: /Users/ADML/Desktop/ML/GenAI/venv/lib/python3.10/site-packages/certifi/cacert.pem

In [ ]:
# Diagnostic - Check environment and paths
import os
from pathlib import Path

print(f"Current working directory: {os.getcwd()}")
print(f"Python executable: {os.sys.executable}")

# Check if .env file exists
env_path = Path("../../.env")
env_path_alt = Path(".env")

print(f"\nChecking .env file paths:")
print(f"../../.env exists: {env_path.exists()}")
print(f"./.env exists: {env_path_alt.exists()}")

if env_path.exists():
    print(f"Content of ../../.env:")
    with open(env_path) as f:
        content = f.read()
        # Hide sensitive info
        lines = content.split('\n')
        for line in lines:
            if line.strip() and '=' in line:
                key, value = line.split('=', 1)
                print(f"  {key}={'*' * len(value.strip('\"'))}")

# Check environment variables
print(f"\nEnvironment variables:")
print(f"HF_TOKEN exists: {'HF_TOKEN' in os.environ}")
print(f"HF_TOKEN value: {'*' * len(os.environ.get('HF_TOKEN', ''))}")

# Try loading dotenv with explicit path
from dotenv import load_dotenv
try:
    # Try loading from parent directory (where .env should be)
    result = load_dotenv(dotenv_path="../../.env")
    print(f"\nload_dotenv('../../.env') result: {result}")
    
    # Check if HF_TOKEN is now available
    print(f"HF_TOKEN after explicit load: {'HF_TOKEN' in os.environ}")
    if 'HF_TOKEN' in os.environ:
        print(f"HF_TOKEN length: {len(os.environ['HF_TOKEN'])}")
except Exception as e:
    print(f"Error loading .env: {e}")

## Feature 4: Text Clustering with Embeddings

Group similar documents together using K-Means clustering.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# Sample texts for visualization
texts_to_visualize = [
    "Machine learning and AI",
    "Deep neural networks",
    "Python programming language",
    "Data science and analytics",
    "Natural language processing",
    "Computer vision applications",
    "Web development with JavaScript",
    "Mobile app development",
    "Cloud computing services",
    "Blockchain technology"
]

# Generate embeddings
print("🎨 Generating embeddings for visualization...")
vis_embeddings = np.array([embeddings.embed_query(text) for text in texts_to_visualize])

# Reduce to 2D using t-SNE
print("🔄 Reducing dimensions with t-SNE...")
tsne = TSNE(n_components=2, random_state=42, perplexity=3)
embeddings_2d = tsne.fit_transform(vis_embeddings)

# Create visualization
plt.figure(figsize=(12, 8))
plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], s=100, alpha=0.6)

# Add labels
for i, text in enumerate(texts_to_visualize):
    plt.annotate(text, (embeddings_2d[i, 0], embeddings_2d[i, 1]),
                xytext=(5, 5), textcoords='offset points', fontsize=9)

plt.title('2D Visualization of Text Embeddings', fontsize=14, fontweight='bold')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✅ Visualization complete!")

In [ ]:
# Install visualization dependencies
!pip install -q matplotlib scikit-learn

## Feature 3: Embedding Visualization with Dimensionality Reduction

Visualize high-dimensional embeddings in 2D space.

In [ ]:
# Perform semantic search
query = "How do I search for similar documents?"
print(f"\n🔍 Searching for: '{query}'\n")

# Search with scores
results_with_scores = vectorstore.similarity_search_with_score(query, k=3)

print("📋 Search Results:")
print("=" * 80)
for i, (doc, score) in enumerate(results_with_scores, 1):
    print(f"\n{i}. Score: {score:.4f}")
    print(f"   {doc.page_content}")

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Extended knowledge base
knowledge_base = [
    "LangChain is a framework for developing applications powered by language models.",
    "Vector embeddings represent text as numerical vectors in high-dimensional space.",
    "Semantic search finds documents based on meaning rather than keyword matching.",
    "FAISS is a library for efficient similarity search developed by Facebook AI.",
    "Transformers are neural network architectures that revolutionized NLP.",
    "GPT (Generative Pre-trained Transformer) models can generate human-like text.",
    "BERT uses bidirectional attention to understand context in both directions.",
    "Fine-tuning adapts pre-trained models to specific tasks with less data.",
    "Embeddings capture semantic relationships between words and sentences.",
    "Vector databases enable fast similarity search over large document collections."
]

# Create FAISS vector store
print("🔧 Creating FAISS vector store...")
vectorstore = FAISS.from_texts(knowledge_base, embeddings)
print("✅ Vector store created with", len(knowledge_base), "documents")

In [ ]:
# Install FAISS
!pip install -q faiss-cpu langchain-community

## Feature 2: Semantic Search with FAISS Vector Store

Build a fast semantic search engine using FAISS.

In [ ]:
def find_most_similar(query_text, documents, top_k=3):
    """Find the most similar documents to a query"""
    # Get query embedding
    query_embedding = embeddings.embed_query(query_text)
    
    # Calculate similarities
    similarities = []
    for i, doc in enumerate(documents):
        doc_embedding = embeddings.embed_query(doc)
        similarity = cosine_similarity([query_embedding], [doc_embedding])[0][0]
        similarities.append((i, doc, similarity))
    
    # Sort by similarity (descending)
    similarities.sort(key=lambda x: x[2], reverse=True)
    
    print(f"\n🔎 Query: '{query_text}'")
    print(f"\n📊 Top {top_k} Most Similar Documents:")
    print("=" * 80)
    for rank, (idx, doc, score) in enumerate(similarities[:top_k], 1):
        print(f"\n{rank}. Similarity: {score:.4f}")
        print(f"   Document {idx}: {doc}")
    
    return similarities[:top_k]

# Test the function
result = find_most_similar("What is artificial intelligence?", documents, top_k=3)

In [ ]:
# Calculate similarity matrix
similarity_matrix = cosine_similarity(doc_embeddings_array)

print("\n🔍 Document Similarity Matrix:")
print("=" * 80)
for i, doc in enumerate(documents):
    print(f"\nDoc {i}: '{doc[:50]}...'")
    print("Similarities:")
    for j, other_doc in enumerate(documents):
        if i != j:
            print(f"  → Doc {j}: {similarity_matrix[i][j]:.4f}")

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Sample documents for analysis
documents = [
    "Machine learning is a subset of artificial intelligence",
    "Deep learning uses neural networks with multiple layers",
    "Python is a popular programming language for data science",
    "Natural language processing helps computers understand human language",
    "AI and machine learning are transforming technology"
]

# Generate embeddings for all documents
print("📄 Generating embeddings for documents...")
doc_embeddings = [embeddings.embed_query(doc) for doc in documents]

# Convert to numpy array for easier computation
doc_embeddings_array = np.array(doc_embeddings)

print(f"✅ Generated embeddings with shape: {doc_embeddings_array.shape}")
print(f"   {len(documents)} documents, {len(doc_embeddings[0])} dimensions")

## Feature 1: Document Similarity Analysis

Compare documents and find the most similar ones using embeddings.